In [1]:
!pip install gymnasium

In [3]:
!pip install gymnasium[classic-control] moviepy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 82.2 MB/s eta 0:00:00


In [16]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo

# 1. 创建环境，必须指定 render_mode="rgb_array"
env = gym.make("CartPole-v1", render_mode="rgb_array")

# 2. 使用 RecordVideo 包装环境
# video_folder 指定视频保存的目录
# name_prefix 指定视频文件的前缀
env = RecordVideo(env, video_folder='./video', name_prefix='cartpole-demo')

# 3. 重置环境并开始交互
state, info = env.reset()
done = False
total_reward = 0

# 运行一个回合直到结束
while not done:
    # 这里我们采用随机动作作为演示，你可以替换为你的智能体模型输出的动作
    action = env.action_space.sample()

    # 交互一步
    next_state, reward, terminated, truncated, info = env.step(action)

    total_reward += reward
    # 判断是否结束 (触发终止条件或达到最大步数截断)
    done = terminated or truncated

# 4. 关闭环境，这一步非常重要！关闭环境时才会将视频缓存完整写入到 MP4 文件中
print(total_reward)
env.close()

print("视频录制完成，已保存在 ./video 目录下")

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /content/video folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


19.0
视频录制完成，已保存在 ./video 目录下


In [7]:
env

<RecordVideo<TimeLimit<OrderEnforcing<PassiveEnvChecker<CartPoleEnv<CartPole-v1>>>>>>

In [8]:
env.action_space

Discrete(2)

In [9]:
env.action_space.sample()

np.int64(1)

In [10]:
env.observation_space

Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)

In [11]:
env.observation_space.low

array([-4.8       ,        -inf, -0.41887903,        -inf], dtype=float32)

In [12]:
env.observation_space.high

array([4.8       ,        inf, 0.41887903,        inf], dtype=float32)

1. env 的输出（环境包装器结构）
输出内容：<RecordVideo<TimeLimit<OrderEnforcing<PassiveEnvChecker<CartPoleEnv<CartPole-v1>>>>>>

含义：这展示了环境的“套娃”结构（Wrappers 包装器）。

最核心的是 CartPoleEnv<CartPole-v1>（推车木杆环境的核心逻辑）。

外面包裹的 PassiveEnvChecker 用于检查代码是否符合 API 标准。

OrderEnforcing 强制规定你必须先调用 reset() 才能调用 step()。

TimeLimit 设置了环境的最大回合步数限制（CartPole-v1 默认限制为 500 步，超时会触发 truncated）。

最外层的 RecordVideo 就是我们之前加进去用于录制视频的包装器。

2. env.action_space (动作空间)
输出内容：Discrete(2)

含义：这定义了你的智能体（Agent）在这个环境中可以执行的动作范围。

Discrete 表示这是一个离散的动作空间（即只能选择特定的整数）。

(2) 表示一共有 2 个合法的动作，索引分别为 0 和 1。

在 CartPole 环境中的具体物理意义：

0：向左推车。

1：向右推车。

3. env.action_space.sample() (动作采样)
输出内容：np.int64(1)

含义：sample() 函数的作用是从上文的动作空间中随机抽取一个合法的动作。这里系统随机抽到了 1（即向右推车），数据类型是 64 位整数。在不知道怎么行动时，我们常用这个方法让智能体进行随机探索。

4. env.observation_space (观测/状态空间)
输出内容：Box([-4.8 -inf -0.41887903 -inf], [4.8 inf 0.41887903 inf], (4,), float32)

含义：这定义了环境每次反馈给智能体的“状态（State）”长什么样。

Box 代表这是一个连续的空间（可以包含任意实数，像一个多维的盒子）。

(4,) 表示每次环境反馈给你的状态是一个包含 4 个浮点数 (float32) 的一维数组。

第一个数组 [-4.8 -inf -0.41887903 -inf] 代表这 4 个数值的下界（最小值），-inf 表示负无穷。

第二个数组 [4.8 inf 0.41887903 inf] 代表这 4 个数值的上界（最大值）。

这 4 个维度在 CartPole 环境中的具体物理意义：

小车的位置 (Cart Position)：范围从 -4.8 到 4.8。

小车的速度 (Cart Velocity)：范围从负无穷到正无穷。

木杆的角度 (Pole Angle)：范围从约 -0.418 弧度到 0.418 弧度（大约是 ±24 度。一旦倾斜超过 12 度，游戏就会判定失败并结束）。

木杆的角速度 (Pole Angular Velocity)：范围从负无穷到正无穷。

1. 维度 0：小车位置 (Cart Position)
low: -4.8

high: 4.8

含义：这是小车在轨道上能被系统观测到的最远物理坐标。

注意（重要）：虽然观测空间的极限是 ±4.8，但在 CartPole 游戏的实际规则中，只要小车的位置超过了 ±2.4（也就是跑出了屏幕可见范围），系统就会判定“游戏失败”并提前终止回合。

2. 维度 1：小车速度 (Cart Velocity)
low: -inf (负无穷大)

high: inf (正无穷大)

含义：小车的移动速度在物理引擎中没有设定上限或下限。它可以无限快（当然在实际训练中会是一个有限的浮点数）。

3. 维度 2：木杆角度 (Pole Angle)
low: -0.41887903 (弧度)

high: 0.41887903 (弧度)

含义：这是木杆倾斜角度的理论极限。0.4188 弧度换算成角度正好是 24 度。

注意（重要）：同样地，24 度只是观测极限。在实际游戏规则中，只要木杆的倾斜角度超过了 ±12度（约 0.209 弧度），系统就会认为木杆已经倒下，判定“游戏失败”并结束回合。

4. 维度 3：木杆角速度 (Pole Angular Velocity)
low: -inf (负无穷大)

high: inf (正无穷大)

含义：木杆倒下的角速度（转动得有多快）同样没有设定上下限。

In [17]:
import glob
from IPython.display import Video, display

# 1. 查找 video 目录下的所有 mp4 文件
video_files = glob.glob('./video/*.mp4')

if len(video_files) > 0:
    # 取最新录制的一个视频
    video_path = video_files[-1]
    print(f"正在播放视频: {video_path}")

    # 删除了 html_attributes 中的 "loop"，保留了 controls（进度条控制）和 autoplay（自动播放）
    display(Video(video_path, embed=True, html_attributes="controls autoplay"))
else:
    print("没有找到视频文件，请检查上面的代码是否正常运行。")

正在播放视频: ./video/cartpole-demo-episode-0.mp4


在强化学习的环境中，render_mode（渲染模式）决定了环境画面以什么形式输出。

'human'（人类模式）：
意思是“渲染给人类看”。当设置这种模式时，代码在运行到 env.render() 时，会在你的电脑桌面上弹出一个真实的图形化游戏窗口，让你能亲眼看到小车和木杆的实时动态。

对比 'rgb_array'（矩阵模式）：
我们在前面将环境保存为视频时，使用的是 render_mode='rgb_array'。这个模式不会弹窗，而是将画面的每一帧转化为计算机能处理的像素矩阵数据，方便我们在后台进行保存或录制。

⚠️ Colab 运行警告：
由于你目前是在 Google Colab 中写代码，Colab 是运行在云端的服务器，它没有物理显示器，不支持直接弹出图形界面。
因此，如果你在 Colab 中将模式设置为 'human' 并运行 env.render()，程序会因为找不到可以用来绘图的显示设备而直接报错崩溃。在 Colab 中，请始终使用 'rgb_array' 并结合 RecordVideo 来查看训练画面。